In [0]:
%sql
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;

In [0]:
%sql
-- Small dimension table: 10 customers.
CREATE TABLE customers AS
SELECT
  id                              AS customer_id,
  concat('Customer_', id)         AS customer_name,
  CASE id % 5
    WHEN 0 THEN 'US' WHEN 1 THEN 'DE' WHEN 2 THEN 'PL'
    WHEN 3 THEN 'JP' ELSE 'BR'
  END                             AS country
FROM range(0, 10);

-- Large fact table: 1,000 orders referencing those 10 customers.
CREATE TABLE orders AS
SELECT
  id                                       AS order_id,
  CAST(rand(1) * 100 AS INT)               AS customer_id,
  CAST(rand(2) * 1000 AS DECIMAL(10,2))    AS amount,
  current_timestamp() - make_interval(0, 0, 0, CAST(rand(3) * 365 AS INT), 0, 0, 0) AS order_ts
FROM range(0, 1000);

-- Make stats explicit so the optimizer has reliable information.
ANALYZE TABLE customers COMPUTE STATISTICS FOR ALL COLUMNS;
ANALYZE TABLE orders    COMPUTE STATISTICS FOR ALL COLUMNS;

SELECT 'orders' AS tbl, count(*) AS rows FROM orders
UNION ALL
SELECT 'customers', count(*) FROM customers;

tbl,rows
orders,1000
customers,10


# Join hints
In this notebook, we demonstrate how to use join hints to control the join strategy.

## Broadcast join hint
Here we're asking the query engine to broadcast the customers (c) table to all partitions with the orders data. You should see a `PhotonBroadcastHashJoin` node in the execution plan.

In [0]:
%sql
EXPLAIN FORMATTED
SELECT /*+ BROADCAST(c) */
  o.order_id, o.amount, c.customer_name, c.country
FROM orders o
JOIN customers c
  ON o.customer_id = c.customer_id;

plan
"== Physical Plan == AdaptiveSparkPlan (10) +- == Initial Plan == PhotonResultStage (9) +- PhotonColumnarToRow (8) +- PhotonProject (7) +- PhotonBroadcastHashJoin Inner (6) :- PhotonScan parquet workspace.default.orders (1) +- PhotonShuffleExchangeSource (5) +- PhotonShuffleMapStage (4) +- PhotonShuffleExchangeSink (3) +- PhotonScan parquet workspace.default.customers (2) (1) PhotonScan parquet workspace.default.orders Output [3]: [order_id#19713L, customer_id#19714, amount#19715] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/db429fdc-4a49-422c-8512-e1d515ade020] ReadSchema: struct RequiredDataFilters: [isnotnull(customer_id#19714)] (2) PhotonScan parquet workspace.default.customers Output [3]: [customer_id#19717L, customer_name#19718, country#19719] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/ede5961b-cb6a-4672-83a4-b3bfea8c673f] ReadSchema: struct RequiredDataFilters: [isnotnull(customer_id#19717L)] (3) PhotonShuffleExchangeSink Input [3]: [customer_id#19717L, customer_name#19718, country#19719] Arguments: SinglePartition (4) PhotonShuffleMapStage Input [3]: [customer_id#19717L, customer_name#19718, country#19719] Arguments: EXECUTOR_BROADCAST, [id=#10833] (5) PhotonShuffleExchangeSource Input [3]: [customer_id#19717L, customer_name#19718, country#19719] (6) PhotonBroadcastHashJoin Left keys [1]: [cast(customer_id#19714 as bigint)] Right keys [1]: [customer_id#19717L] Join type: Inner Join condition: None (7) PhotonProject Input [6]: [order_id#19713L, customer_id#19714, amount#19715, customer_id#19717L, customer_name#19718, country#19719] Arguments: [order_id#19713L, amount#19715, customer_name#19718, country#19719] (8) PhotonColumnarToRow Input [4]: [order_id#19713L, amount#19715, customer_name#19718, country#19719] (9) PhotonResultStage Input [4]: [order_id#19713L, amount#19715, customer_name#19718, country#19719] (10) AdaptiveSparkPlan Output [4]: [order_id#19713L, amount#19715, customer_name#19718, country#19719] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = customers, orders"


## Sort-merge join hint
Here we'll use a sort-merge join hint to instruct the query engine to use a sort-merge join strategy. This is typically chosen when both tables are large and already sorted or can be efficiently sorted on the join keys.

The execution plan first shuffles both tables by the join column (customer_id) and later sorts them using the same column:
```
(6) PhotonSort
Input [4]: [order_id#15670L, customer_id#15671, amount#15672, _sort_order0#15688L]
Arguments: [cast(customer_id#15671 as bigint) ASC NULLS FIRST]
```

In the end, the query engine performs the sort-merge join:
```
(17) SortMergeJoin
Left keys [1]: [cast(customer_id#15671 as bigint)]
Right keys [1]: [customer_id#15674L]
Join type: Inner
Join condition: None
```

`SHUFFLE_MERGE` and `MERGEJOIN` are two aliases for the `MERGE` hint. (cf. https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-select-hints#syntax-2)

In [0]:
%sql
EXPLAIN FORMATTED
SELECT /*+ MERGE(o, c) */
  o.order_id, o.amount, c.customer_name, c.country
FROM orders o
JOIN customers c
  ON o.customer_id = c.customer_id;

plan


## Shuffle hash join hint
Here we'll use a shuffle hash join hint to instruct the query engine to use a shuffle hash join strategy. This is typically chosen when both tables are large and can be efficiently shuffled on the join keys.

The execution plan first shuffles both tables by the join column (customer_id) and later applies a hash join using the same column:
```
(2) PhotonShuffleExchangeSink
Input [3]: [order_id#15708L, customer_id#15709, amount#15710]
Arguments: hashpartitioning(cast(customer_id#15709 as bigint), 16)
```

In the end, it uses the shuffle join strategy:
```
(9) PhotonShuffledHashJoin
Left keys [1]: [cast(customer_id#15709 as bigint)]
Right keys [1]: [customer_id#15712L]
Join type: Inner
Join condition: None
```

In [0]:
%sql
EXPLAIN FORMATTED
SELECT /*+ SHUFFLE_HASH(c) */
  o.order_id, o.amount, c.customer_name
FROM orders o
JOIN customers c
  ON o.customer_id = c.customer_id;

plan
"== Physical Plan == AdaptiveSparkPlan (13) +- == Initial Plan == PhotonResultStage (12) +- PhotonColumnarToRow (11) +- PhotonProject (10) +- PhotonShuffledHashJoin Inner (9) :- PhotonShuffleExchangeSource (4) : +- PhotonShuffleMapStage (3) : +- PhotonShuffleExchangeSink (2) : +- PhotonScan parquet workspace.default.orders (1) +- PhotonShuffleExchangeSource (8) +- PhotonShuffleMapStage (7) +- PhotonShuffleExchangeSink (6) +- PhotonScan parquet workspace.default.customers (5) (1) PhotonScan parquet workspace.default.orders Output [3]: [order_id#19784L, customer_id#19785, amount#19786] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/db429fdc-4a49-422c-8512-e1d515ade020] ReadSchema: struct RequiredDataFilters: [isnotnull(customer_id#19785)] (2) PhotonShuffleExchangeSink Input [3]: [order_id#19784L, customer_id#19785, amount#19786] Arguments: hashpartitioning(cast(customer_id#19785 as bigint), 16) (3) PhotonShuffleMapStage Input [3]: [order_id#19784L, customer_id#19785, amount#19786] Arguments: ENSURE_REQUIREMENTS, [id=#11036] (4) PhotonShuffleExchangeSource Input [3]: [order_id#19784L, customer_id#19785, amount#19786] (5) PhotonScan parquet workspace.default.customers Output [2]: [customer_id#19791L, customer_name#19792] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/ede5961b-cb6a-4672-83a4-b3bfea8c673f] ReadSchema: struct RequiredDataFilters: [isnotnull(customer_id#19791L)] (6) PhotonShuffleExchangeSink Input [2]: [customer_id#19791L, customer_name#19792] Arguments: hashpartitioning(customer_id#19791L, 16) (7) PhotonShuffleMapStage Input [2]: [customer_id#19791L, customer_name#19792] Arguments: ENSURE_REQUIREMENTS, [id=#11042] (8) PhotonShuffleExchangeSource Input [2]: [customer_id#19791L, customer_name#19792] (9) PhotonShuffledHashJoin Left keys [1]: [cast(customer_id#19785 as bigint)] Right keys [1]: [customer_id#19791L] Join type: Inner Join condition: None (10) PhotonProject Input [5]: [order_id#19784L, customer_id#19785, amount#19786, customer_id#19791L, customer_name#19792] Arguments: [order_id#19784L, amount#19786, customer_name#19792] (11) PhotonColumnarToRow Input [3]: [order_id#19784L, amount#19786, customer_name#19792] (12) PhotonResultStage Input [3]: [order_id#19784L, amount#19786, customer_name#19792] (13) AdaptiveSparkPlan Output [3]: [order_id#19784L, amount#19786, customer_name#19792] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = customers, orders"


## Shuffle replication nested loop hint
Here we'll use a shuffle replication nested loop join hint to instruct the query engine to use a shuffle replication nested loop join strategy. This is typically chosen when one table is small and the other is large, allowing efficient replication and nested loop processing across partitions.

The execution plan creates a cartesian product of two input tables:
```
(7) CartesianProduct
Join type: Inner
Join condition: ((customer_id#15748L >= cast((customer_id#15745 - 2) as bigint)) AND (customer_id#15748L <= cast((customer_id#15745 + 2) as bigint)))
```

In [0]:
%sql
EXPLAIN FORMATTED
SELECT /*+ SHUFFLE_REPLICATE_NL(c) */
  o.order_id, c.customer_name
FROM orders o
JOIN customers c
  ON c.customer_id BETWEEN o.customer_id - 2 AND o.customer_id + 2
WHERE o.order_id < 100;   -- keep the result sane; this is O(N*M)

plan
"== Physical Plan == * Project (8) +- CartesianProduct Inner (7) :- * ColumnarToRow (3) : +- PhotonResultStage (2) : +- PhotonScan parquet workspace.default.orders (1) +- * ColumnarToRow (6) +- PhotonResultStage (5) +- PhotonScan parquet workspace.default.customers (4) (1) PhotonScan parquet workspace.default.orders Output [2]: [order_id#15744L, customer_id#15745] DictionaryFilters: [(order_id#15744L < 100)] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/cb29b555-dd2c-4052-94b1-f0e74d2f8f37] ReadSchema: struct RequiredDataFilters: [isnotnull(customer_id#15745), isnotnull(order_id#15744L), (order_id#15744L < 100)] (2) PhotonResultStage Input [2]: [order_id#15744L, customer_id#15745] (3) ColumnarToRow [codegen id : 1] Input [2]: [order_id#15744L, customer_id#15745] (4) PhotonScan parquet workspace.default.customers Output [2]: [customer_id#15748L, customer_name#15749] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/1daeccf8-0794-4bcf-9262-9094e38b8fea] ReadSchema: struct RequiredDataFilters: [isnotnull(customer_id#15748L)] (5) PhotonResultStage Input [2]: [customer_id#15748L, customer_name#15749] (6) ColumnarToRow [codegen id : 2] Input [2]: [customer_id#15748L, customer_name#15749] (7) CartesianProduct Join type: Inner Join condition: ((customer_id#15748L >= cast((customer_id#15745 - 2) as bigint)) AND (customer_id#15748L <= cast((customer_id#15745 + 2) as bigint))) (8) Project [codegen id : 3] Output [2]: [order_id#15744L, customer_name#15749] Input [4]: [order_id#15744L, customer_id#15745, customer_id#15748L, customer_name#15749] == Photon Explanation == Photon does not fully support the query because: Unsupported node: CartesianProduct Inner, ((customer_id#15748L >= cast((customer_id#15745 - 2) as bigint)) AND (customer_id#15748L <= cast((customer_id#15745 + 2) as bigint))). Reference node: CartesianProduct Inner, ((customer_id#15748L >= cast((customer_id#15745 - 2) as bigint)) AND (customer_id#15748L <= cast((customer_id#15745 + 2) as bigint))) == Optimizer Statistics (table names per statistics state) == missing = partial = full = customers, orders"
